# Notebook 04 — Post-Training Quantization (PTQ) for LLM Inference  
**GPU-first (Colab), CPU fallback** — compare **FP16 vs 8-bit vs 4-bit** on **memory + speed + perplexity**

This notebook is designed as a **realistic deployment-style** experiment:

- Load an open small chat LLM (default: **TinyLlama-1.1B-Chat**).
- Build **three inference variants**:
  1. **FP16** (baseline on GPU)
  2. **8-bit** quantized with **bitsandbytes**
  3. **4-bit** (NF4) quantized with **bitsandbytes**
- Benchmark:
  - **GPU memory** (peak allocated)
  - **Speed** (tokens/sec)
  - **Quality proxy**: **perplexity** on a small slice of Wikitext-2
- Provide **CPU fallback**:
  - If CUDA or bitsandbytes is not available, use **PyTorch dynamic quantization** on CPU.

Why this matters:
- PTQ is one of the main practical routes to deploy foundation models under memory/latency constraints.
- It often composes well with other techniques (distillation, pruning, LoRA/QLoRA).


## 1) Install dependencies

We use:
- `transformers`, `accelerate`: model loading + generation
- `datasets`: Wikitext-2 evaluation data
- `bitsandbytes`: 8-bit / 4-bit quantization on GPU (optional, best effort)
- `psutil`: CPU memory helpers


In [1]:
import sys, subprocess

def pip_install(pkgs):
    print("Installing:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install([
    "numpy>=1.24",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "datasets>=2.18",
    "transformers>=4.41",
    "accelerate>=0.30",
    "psutil>=5.9",
])

# bitsandbytes is optional; install it if possible
try:
    pip_install(["bitsandbytes>=0.43"])
    BNB_AVAILABLE = True
except Exception as e:
    print("bitsandbytes install failed (this is OK on CPU-only runtimes).")
    print("Reason:", str(e)[:200])
    BNB_AVAILABLE = False

print("Done.")


Installing: ['numpy>=1.24', 'pandas>=2.0', 'matplotlib>=3.7', 'datasets>=2.18', 'transformers>=4.41', 'accelerate>=0.30', 'psutil>=5.9']
Installing: ['bitsandbytes>=0.43']
Done.


## 2) Imports + device selection (GPU-first)

We detect CUDA. If available, we will attempt 8-bit and 4-bit loads with bitsandbytes.


In [2]:
import os, time, math, gc, random
import numpy as np
import pandas as pd

import torch
import psutil

from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU RAM: {props.total_memory/(1024**3):.1f} GB")
print("bitsandbytes available:", BNB_AVAILABLE)


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU RAM: 39.6 GB
bitsandbytes available: True


## 3) Choose a model (open + realistic)

Default: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

Why:
- Small enough for most Colab GPUs in FP16.
- Quantization variants show meaningful trade-offs.

You may uncomment larger models if you have enough VRAM.


In [3]:
MODEL_CANDIDATES = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # Uncomment for heavier experiments:
    # "Qwen/Qwen2.5-3B-Instruct",
    # "mistralai/Mistral-7B-Instruct-v0.2",
]

MODEL_NAME = MODEL_CANDIDATES[0]
print("Selected:", MODEL_NAME)


Selected: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Helpers: parameter count, memory reporting, safe cleanup


In [4]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def cpu_rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

def gpu_mem_mb():
    if not torch.cuda.is_available():
        return None, None
    return (torch.cuda.memory_allocated() / (1024**2),
            torch.cuda.memory_reserved() / (1024**2))

def cleanup(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("CPU RSS (MB):", round(cpu_rss_mb(), 1))
if DEVICE.type == "cuda":
    a, r = gpu_mem_mb()
    print("GPU allocated/reserved (MB):", round(a,1), "/", round(r,1))


CPU RSS (MB): 798.7
GPU allocated/reserved (MB): 0.0 / 0.0


## 5) Load tokenizer (shared across variants)


In [5]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Tokenizer vocab size:", tok.vocab_size)
print("Pad token:", tok.pad_token, "EOS token:", tok.eos_token)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer vocab size: 32000
Pad token: </s> EOS token: </s>


## 6) Load three variants (FP16 / 8-bit / 4-bit)

We attempt:
- FP16 baseline (GPU) or FP32 (CPU)
- 8-bit and 4-bit via bitsandbytes (GPU only)

If loading fails, the variant is skipped.


In [6]:
from transformers import BitsAndBytesConfig

def load_fp16_or_fp32():
    if DEVICE.type == "cuda":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cpu")
    model.eval()
    return model

def load_8bit():
    if not (DEVICE.type == "cuda" and BNB_AVAILABLE):
        return None
    qconf = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=qconf,
        device_map="auto",
    )
    model.eval()
    return model

def load_4bit_nf4():
    if not (DEVICE.type == "cuda" and BNB_AVAILABLE):
        return None
    qconf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=qconf,
        device_map="auto",
    )
    model.eval()
    return model

variants = {}

print("Loading FP16/FP32 baseline...")
variants["fp16_or_fp32"] = load_fp16_or_fp32()

if DEVICE.type == "cuda":
    print("Loading 8-bit...")
    try:
        variants["int8"] = load_8bit()
    except Exception as e:
        print("8-bit load failed:", str(e)[:200])
        variants["int8"] = None

    print("Loading 4-bit (NF4)...")
    try:
        variants["int4_nf4"] = load_4bit_nf4()
    except Exception as e:
        print("4-bit load failed:", str(e)[:200])
        variants["int4_nf4"] = None

for k, m in variants.items():
    print(k, "=>", None if m is None else f"{count_params(m)/1e6:.1f}M params on {m.device}")


Loading FP16/FP32 baseline...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading 8-bit...
Loading 4-bit (NF4)...
fp16_or_fp32 => 1100.0M params on cuda:0
int8 => 1100.0M params on cuda:0
int4_nf4 => 615.6M params on cuda:0


## 7) Quick generation sanity check


In [7]:
def generate(model, prompt, max_new_tokens=128):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

prompt = "Explain in 3 bullet points what post-training quantization is."
for name, model in variants.items():
    if model is None:
        continue
    print("\n=== Variant:", name, "device:", model.device, "===")
    print(generate(model, prompt, max_new_tokens=96)[:600])



=== Variant: fp16_or_fp32 device: cuda:0 ===
Explain in 3 bullet points what post-training quantization is.

=== Variant: int8 device: cuda:0 ===
Explain in 3 bullet points what post-training quantization is.

=== Variant: int4_nf4 device: cuda:0 ===
Explain in 3 bullet points what post-training quantization is.


## 8) Benchmark 1 — GPU memory (peak allocated)


In [8]:
def peak_gpu_mb_during_generate(model, prompt, max_new_tokens=128):
    if DEVICE.type != "cuda":
        return None
    torch.cuda.reset_peak_memory_stats()
    _ = generate(model, prompt, max_new_tokens=max_new_tokens)
    torch.cuda.synchronize()
    return float(torch.cuda.max_memory_allocated() / (1024**2))

mem_rows = []
for name, model in variants.items():
    if model is None:
        continue
    mem_rows.append({
        "variant": name,
        "device": str(model.device),
        "params_M": count_params(model)/1e6,
        "peak_gpu_MB": peak_gpu_mb_during_generate(model, prompt, max_new_tokens=128),
    })

mem_df = pd.DataFrame(mem_rows)
mem_df


,variant,device,params_M,peak_gpu_MB
0,fp16_or_fp32,cuda:0,1100.048384,4037.814453
1,int8,cuda:0,1100.048384,4038.040039
2,int4_nf4,cuda:0,615.606272,4060.158203


## 9) Benchmark 2 — Speed (tokens/sec)


In [9]:
def benchmark_tokens_per_sec(model, prompts, max_new_tokens=128, warmup=1, iters=3):
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        for _ in range(warmup):
            _ = model.generate(
                **enc, max_new_tokens=max_new_tokens,
                do_sample=False, num_beams=1,
                pad_token_id=tok.eos_token_id
            )
        if model.device.type == "cuda":
            torch.cuda.synchronize()

    start = time.time()
    total_new = 0
    with torch.no_grad():
        for _ in range(iters):
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens,
                do_sample=False, num_beams=1,
                pad_token_id=tok.eos_token_id
            )
            new_tokens = (out.shape[1] - enc["input_ids"].shape[1]) * out.shape[0]
            total_new += max(0, int(new_tokens))
        if model.device.type == "cuda":
            torch.cuda.synchronize()
    elapsed = time.time() - start
    return total_new / max(elapsed, 1e-9)

BATCH_PROMPTS = 8
GEN_MAX_NEW_TOKENS = 128

prompts = [
    "Summarize the main idea of knowledge distillation in 2 sentences.",
    "Explain what RAG is and why retrieval helps.",
    "Write a short checklist for evaluating a RAG system.",
    "What is the difference between PTQ and QAT?",
    "Give 3 deployment risks of LLMs and mitigations.",
    "Explain BF16 vs FP16 in one paragraph.",
    "List 3 benefits and 3 risks of 4-bit quantization.",
    "Explain why metadata filtering matters in enterprise RAG.",
][:BATCH_PROMPTS]

speed_rows = []
for name, model in variants.items():
    if model is None:
        continue
    try:
        tps = benchmark_tokens_per_sec(model, prompts, max_new_tokens=GEN_MAX_NEW_TOKENS, warmup=1, iters=3)
        speed_rows.append({"variant": name, "device": str(model.device), "tokens_per_sec": float(tps)})
    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg and torch.cuda.is_available():
            print("OOM during speed benchmark for", name, "- reduce BATCH_PROMPTS or GEN_MAX_NEW_TOKENS.")
        else:
            print("Error for", name, ":", str(e)[:200])

speed_df = pd.DataFrame(speed_rows).sort_values("tokens_per_sec", ascending=False)
speed_df


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

,variant,device,tokens_per_sec
0,fp16_or_fp32,cuda:0,214.921014
2,int4_nf4,cuda:0,101.626321
1,int8,cuda:0,61.666155


## 10) Benchmark 3 — Quality proxy: perplexity on Wikitext-2 (small slice)


In [10]:
from datasets import load_dataset

wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
lines = [x["text"] for x in wikitext if x["text"].strip()][:200]
eval_text = "\n".join(lines)

print("Eval chars:", len(eval_text))
print(eval_text[:250])


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Eval chars: 100900
 = Robert Boulter = 

 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons written by Simon Stephens ,


In [11]:
def perplexity(model, text, block_size=256):
    model.eval()
    enc = tok(text, return_tensors="pt")
    input_ids = enc["input_ids"][0]
    nlls = []
    for i in range(0, input_ids.numel(), block_size):
        block = input_ids[i:i+block_size]
        if block.numel() < 2:
            continue
        block = block.unsqueeze(0).to(model.device)
        with torch.no_grad():
            out = model(block, labels=block)
            nlls.append(float(out.loss.detach().cpu()))
    if not nlls:
        return None
    return float(math.exp(sum(nlls) / len(nlls)))

ppl_rows = []
for name, model in variants.items():
    if model is None:
        continue
    try:
        ppl = perplexity(model, eval_text, block_size=256)
        ppl_rows.append({"variant": name, "device": str(model.device), "perplexity": ppl})
        print(name, "ppl:", ppl)
    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg and torch.cuda.is_available():
            print("OOM during perplexity for", name, "- try smaller block_size.")
        else:
            print("Error for", name, ":", str(e)[:200])

ppl_df = pd.DataFrame(ppl_rows).sort_values("perplexity")
ppl_df


Token indices sequence length is longer than the specified maximum sequence length for this model (27660 > 2048). Running this sequence through the model will result in indexing errors


fp16_or_fp32 ppl: 12.809175946823617
int8 ppl: 12.847315610226987
int4_nf4 ppl: 13.62716252126


,variant,device,perplexity
0,fp16_or_fp32,cuda:0,12.809176
1,int8,cuda:0,12.847316
2,int4_nf4,cuda:0,13.627163


## 11) CPU fallback: dynamic quantization (Linear → int8)

If you are on CPU-only runtime (or bitsandbytes fails), you can still quantize using:
- `torch.quantization.quantize_dynamic`

This typically reduces model size and can speed up CPU inference (depending on CPU and ops).


In [12]:
CPU_VARIANTS = {}

def load_cpu_fp32():
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cpu")
    m.eval()
    return m

def dynamic_int8_quantize_cpu(model_cpu):
    import torch.nn as nn
    qmodel = torch.quantization.quantize_dynamic(
        model_cpu, {nn.Linear}, dtype=torch.qint8
    )
    qmodel.eval()
    return qmodel

if DEVICE.type == "cpu":
    print("CPU-only mode: building CPU fp32 and dynamic int8 variants...")
    m_cpu = load_cpu_fp32()
    CPU_VARIANTS["cpu_fp32"] = m_cpu
    try:
        CPU_VARIANTS["cpu_int8_dynamic"] = dynamic_int8_quantize_cpu(load_cpu_fp32())
        print("Dynamic int8 quantization OK.")
    except Exception as e:
        print("Dynamic int8 quantization failed:", str(e)[:200])

    for k, m in CPU_VARIANTS.items():
        print(k, "params:", f"{count_params(m)/1e6:.1f}M")
else:
    print("CUDA available: CPU fallback is optional; run it only if you need CPU-only deployment numbers.")


CUDA available: CPU fallback is optional; run it only if you need CPU-only deployment numbers.


## 12) Consolidated report table


In [13]:
report = mem_df.merge(speed_df, on=["variant","device"], how="left").merge(ppl_df, on=["variant","device"], how="left")
report


,variant,device,params_M,peak_gpu_MB,tokens_per_sec,perplexity
0,fp16_or_fp32,cuda:0,1100.048384,4037.814453,214.921014,12.809176
1,int8,cuda:0,1100.048384,4038.040039,61.666155,12.847316
2,int4_nf4,cuda:0,615.606272,4060.158203,101.626321,13.627163


## 13) Interpretation

Typical outcomes:
- **4-bit NF4** uses the least GPU memory and is often fast.
- **8-bit** is a middle ground: smaller memory than FP16, small quality change.
- **FP16** often yields best perplexity (proxy) but uses most memory.

Next step:
- Quantize your **RAG generator** (Notebook 02) and measure end-to-end latency/quality.


## 14) Optional extensions (advanced)

1. **QLoRA** fine-tuning: keep 4-bit weights, train adapters to recover quality.
2. **Calibration for PTQ**: use representative data for better quantization ranges.
3. Compare methods: GPTQ/AWQ (more complex).
4. Multi-objective selection: choose variant by (perplexity, tokens/sec, memory).
